In [77]:
%load_ext autoreload
%autoreload 2

import time
import numpy as np
import mujoco.viewer
from hexapod_env import HexapodEnv

from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import SubprocVecEnv, VecNormalize
from stable_baselines3.common.callbacks import BaseCallback, CheckpointCallback

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [78]:
N_ENVS = 8
TOTAL_STEPS = 1_000_000
RUN_NAME = "stage2.1-slip"
BASE = "models/stage2-tripod"
env = HexapodEnv()

In [79]:
def make_env():
    return Monitor(HexapodEnv())

vec_env = SubprocVecEnv([make_env for _ in range(N_ENVS)])
vec_env = VecNormalize.load(f"{BASE}/vecnormalize.pkl", vec_env)
vec_env.training = True
vec_env.norm_reward = True

In [ ]:
INFO_KEYS = (
    "forward_vel",
    "side_vel",
    "reward",
    "r_vel",
    "r_air",
    "r_yaw",
    "r_gait",
    "pen_yaw",
    "pen_tilt",
    "pen_rate",
    "pen_vy",
    "pen_vz",
)

class InfoCallback(BaseCallback):
    def _on_step(self):
        for info in self.locals["infos"]:
            for k in INFO_KEYS:
                self.logger.record_mean("env/" + k, info[k])
        return True

In [81]:
# model = PPO(
#     "MlpPolicy",
#     vec_env,
#     n_steps=1024,
#     batch_size=2048,
#     n_epochs=5,
#     learning_rate=lambda progress: 3e-4 * progress,
#     gamma=0.99,
#     gae_lambda=0.95,
#     clip_range=0.2,
#     ent_coef=0.0,
#     policy_kwargs=dict(
#         net_arch=dict(pi=[256, 256], vf=[256, 256]),
#         log_std_init=-1.0,
#     ),
#     tensorboard_log="logs/",
#     device="cpu",
#     verbose=1,
# )

# checkpoint = CheckpointCallback(
#     save_freq=100_000 // N_ENVS,
#     save_path=f"models/{RUN_NAME}/checkpoints",
#     save_vecnormalize=True,
# )

In [82]:
lr = lambda progress: 1e-4 * progress

model = PPO.load(
    f"{BASE}/model",
    env=vec_env,
    device="cpu",
    tensorboard_log="logs/",
    custom_objects={"learning_rate": lr, "lr_schedule": lr},
)

checkpoint = CheckpointCallback(
    save_freq=100_000 // N_ENVS,
    save_path=f"models/{RUN_NAME}/checkpoints",
    save_vecnormalize=True,
)

In [83]:
model.learn(total_timesteps=TOTAL_STEPS, callback=[checkpoint, InfoCallback()], tb_log_name=RUN_NAME)

model.save(f"models/{RUN_NAME}/model")
vec_env.save(f"models/{RUN_NAME}/vecnormalize.pkl")
vec_env.close()

Logging to logs/stage2.1-slip_1
height=0.1134, gravity_z=-0.9940, terminated=False, truncated=True
height=0.1182, gravity_z=-0.9991, terminated=False, truncated=True
height=0.1181, gravity_z=-0.9984, terminated=False, truncated=True
height=0.1143, gravity_z=-0.9995, terminated=False, truncated=True
height=0.1105, gravity_z=-0.9928, terminated=False, truncated=True
height=0.1180, gravity_z=-0.9988, terminated=False, truncated=True
height=0.1089, gravity_z=-0.9948, terminated=False, truncated=True
height=0.1143, gravity_z=-0.9999, terminated=False, truncated=True
---------------------------------
| env/               |          |
|    forward_vel     | 0.147    |
|    pen_rate        | 5.42     |
|    pen_slip        | 0.0796   |
|    pen_tilt        | 0.00338  |
|    pen_vy          | 0.00137  |
|    pen_vz          | 0.00252  |
|    pen_yaw         | 0.0109   |
|    r_air           | 0.0183   |
|    r_gait          | 0.895    |
|    r_vel           | 0.947    |
|    r_yaw           | 0